## Initialize Phoenix and Load Dataset


In [ ]:
from phoenix.client import Client
from getpass import getpass

phoenix_base_url = input("Enter your Phoenix OTLP endpoint (e.g., https://your-phoenix.com): ").strip("/ ")
phoenix_api_key = getpass("Enter your Phoenix API key (hidden): ").strip()
phoenix_client = Client(base_url=phoenix_base_url, api_key=phoenix_api_key)

In [ ]:

from openinference.instrumentation.dspy import DSPyInstrumentor
from opentelemetry import trace as trace_api
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk import trace as trace_sdk
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.resources import Resource


resource = Resource.create({"service.name": "spam-filter-notebook"})

tracer_provider = trace_sdk.TracerProvider(resource=resource)
headers = {"authorization": f"Bearer {phoenix_api_key}"}

otlp_exporter = OTLPSpanExporter(endpoint=phoenix_base_url + "/v1/traces", headers=headers)

tracer_provider.add_span_processor(SimpleSpanProcessor(otlp_exporter))
trace_api.set_tracer_provider(tracer_provider)

# Instrument DSPy
DSPyInstrumentor().instrument()

In [ ]:
DSPyInstrumentor().uninstrument()

In [ ]:
dataset = phoenix_client.datasets.get_dataset(dataset="spam-classification", timeout=300)

In [ ]:
dataset.examples[0]

## Setup DSPY Language Models


In [ ]:
from getpass import getpass

openrouter_api_key = getpass("Enter your OpenRouter API key (hidden): ").strip()

#### 1. Import DSPY and Configure Language Models

In [ ]:
import dspy

llm_base_url = "https://openrouter.ai/api/v1"

task_lm = dspy.LM("openrouter/google/gemini-2.5-flash", api_base=llm_base_url, reasoning_effort="disable", api_key=openrouter_api_key)
prompt_lm = dspy.LM("openrouter/anthropic/claude-sonnet-4.5", api_base=llm_base_url, api_key=openrouter_api_key)  # Stronger model for optimization

dspy.configure(lm=task_lm, adapter=dspy.JSONAdapter())

#### 2. Define Spam Classification Signature

In [ ]:
from typing import Literal

SpamLevel = Literal[
    "not_spam",
    "unlikely_spam",
    "suspicious",
    "likely_spam",
    "very_likely_spam",
    "definitely_spam",
]


class SpamClassification(dspy.Signature):
    """Analyze an email message and determine if it is spam.

    You are a cybersecurity email analyst specializing in spam detection.
    """

    text: str = dspy.InputField(desc="The plaintext body of the email message to analyze")
    subject: str = dspy.InputField(desc="The subject line key content of the email")
    return_path: str | None = dspy.InputField(desc="The Return-Path header address, indicating where bounces are sent")
    from_address: str | None = dspy.InputField(desc="The visible From header address shown to the recipient")
    received_spf: str | None = dspy.InputField(
        desc="The SPF verification result (e.g., Pass, Fail, SoftFail) extracted from headers"
    )
    reply_address: str | None = dspy.InputField(desc="The Reply-To address if different from the sender")
    authentication_results: str | None = dspy.InputField(
        desc="Technical authentication results (DKIM, SPF, DMARC) found in the headers"
    )
    reasons: list[str] = dspy.OutputField(
        desc="A list of specific observations or red flags justifying the classification"
    )
    classification: SpamLevel = dspy.OutputField(desc="The final determination of the spam risk level")


#### 3. Define Spam Classification Module

In [ ]:
class SpamClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.classify = dspy.Predict(SpamClassification)

    def forward(
        self,
        text: str,
        subject: str,
        return_path: str | None = None,
        from_address: str | None = None,
        received_spf: str | None = None,
        reply_address: str | None = None,
        authentication_results: str | None = None,
    ) -> SpamClassification:
        return self.classify(
            text=text,
            subject=subject,
            return_path=return_path,
            from_address=from_address,
            received_spf=received_spf,
            reply_address=reply_address,
            authentication_results=authentication_results,
        )

#### 4. Prepare DSPY Dataset

In [ ]:
trainset = []

for example in dataset.examples:
    inputs = {
            "text": example["input"]["text"],
            "subject": example["input"]["subject"],
            "return_path": example["input"].get("return_path"),
            "from_address": example["input"].get("from_address"),
            "received_spf": example["input"].get("received_spf"),
            "reply_address": example["input"].get("reply_address"),
            "authentication_results": example["input"].get("authentication_results"),
        }
    outputs = {
            "reasons": [],
            "classification": "definitely_spam" if example["output"]["is_spam"] else "not_spam",
        }
    trainset.append(dspy.Example(**inputs, **outputs).with_inputs(*inputs.keys()))

#### 5. Define Metric

In [ ]:
LEVEL_MAP = {
    "not_spam": 0,
    "unlikely_spam": 1,
    "suspicious": 2,
    "likely_spam": 3,
    "very_likely_spam": 4,
    "definitely_spam": 5,
}


def spam_metric(example, prediction, trace=None) -> float:
    """
    Evaluate spam classification using binary ground truth.

    Scoring breakdown:
    - 70%: Binary decision correctness (matches production threshold)
    - 30%: Calibration bonus (rewards confident correct predictions)

    Asymmetric penalties:
    - False positive (ham → junk): 0.0 (worst - loses legitimate email)
    - False negative (spam → inbox): 0.3 (bad but recoverable)
    """
    pred_level = LEVEL_MAP.get(prediction.classification, 3)
    is_spam = example.classification == "definitely_spam"

    pred_is_spam = pred_level > 3  # Threshold for junking

    if pred_is_spam == is_spam:
        decision_score = 1.0  # Correct decision
    elif pred_is_spam and not is_spam:
        decision_score = 0.0  # False positive - unacceptable
    else:
        decision_score = 0.3  # False negative - tolerable

    # Calibration: reward confident correct predictions
    # Spam should score high (5), ham should score low (0)
    if is_spam:
        calibration = pred_level / 5  # Higher = better for spam
    else:
        calibration = 1 - (pred_level / 5)  # Lower = better for ham

    return 0.7 * decision_score + 0.3 * calibration

#### 6. Optimize Classifier with DSPY

In [ ]:
optimizer = dspy.MIPROv2(
    metric=spam_metric,
    auto="medium",
    num_threads=25,
    prompt_model=task_lm,
    task_model=prompt_lm,
)

In [ ]:
optimized = optimizer.compile(
    SpamClassifier(),
    trainset=trainset,
)